In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 11.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=b08cbf86bcf3dbb2be915e61606948fe32edd8094e19503ce1891b976fe47c0d
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# Import Qiskit
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

backend = BasicSimulator()

# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.



In [3]:
# Define quantum_random_bit() to replace Python random

def quantum_random_bit():
    """
    Generate one random bit using quantum measurement.
    Start with |0>, apply H to create 1/sqrt(2)(|0> + |1>),
    then measure.
    """
    qc = QuantumCircuit(1, 1)
    qc.h(0)
    qc.measure(0, 0)

    tqc = transpile(qc, backend)
    result = backend.run(tqc, shots=1).result()
    counts = result.get_counts()

    bit = list(counts.keys())[0]
    return int(bit)


def quantum_random_bits(n):
    """
    Generate n quantum random bits.
    """
    bits = []

    for _ in range(n):
        bits.append(quantum_random_bit())

    return bits

In [4]:
def prepare_and_measure(alice_bit, alice_basis, bob_basis):
    """
    Alice prepares one qubit.
    Bob measures it.

    basis = 0 means Z basis
    basis = 1 means X basis
    """

    qc = QuantumCircuit(1, 1)

    # -------------------------
    # Alice prepares the qubit
    # -------------------------

    # If Alice's bit is 1, change |0> to |1>
    if alice_bit == 1:
        qc.x(0)

    # If Alice uses X basis, apply H
    # bit 0 becomes |+>
    # bit 1 becomes |->
    if alice_basis == 1:
        qc.h(0)

    # -------------------------
    # Bob measures the qubit
    # -------------------------

    # If Bob measures in X basis, apply H before measurement
    if bob_basis == 1:
        qc.h(0)

    # measures the qubit and stores the result in the classical bit
    qc.measure(0, 0)

    tqc = transpile(qc, backend)

    # runs the circuit one time
    result = backend.run(tqc, shots=1).result()
    counts = result.get_counts()

    # returns Bob's measured bit
    measured_bit = list(counts.keys())[0]
    return int(measured_bit)

In [10]:
n_qubits = 15

# Alice randomly chooses bits
alice_bits = quantum_random_bits(n_qubits)

# Alice randomly chooses bases
# 0 = Z basis
# 1 = X basis
alice_bases = quantum_random_bits(n_qubits)

print("Alice bits:")
print(alice_bits)

print("Alice bases:")
print(alice_bases)

Alice bits:
[1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0]
Alice bases:
[1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1]


In [11]:
# Bob randomly chooses measurement bases
# 0 = Z basis
# 1 = X basis
bob_bases = quantum_random_bits(n_qubits)

bob_results = []

for i in range(n_qubits):
    result = prepare_and_measure(
        alice_bit=alice_bits[i],
        alice_basis=alice_bases[i],
        bob_basis=bob_bases[i]
    )

    bob_results.append(result)

print("Bob bases:")
print(bob_bases)

print("Bob results:")
print(bob_results)

Bob bases:
[1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0]
Bob results:
[1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0]


In [12]:
matching_indices = []

sifted_alice_key = []
sifted_bob_key = []

for i in range(n_qubits):
    if alice_bases[i] == bob_bases[i]:
        matching_indices.append(i)
        sifted_alice_key.append(alice_bits[i])
        sifted_bob_key.append(bob_results[i])

print("Matching basis indices:")
print(matching_indices)

print("Sifted Alice key:")
print(sifted_alice_key)

print("Sifted Bob key:")
print(sifted_bob_key)

Matching basis indices:
[0, 3, 9, 10]
Sifted Alice key:
[1, 0, 0, 0]
Sifted Bob key:
[1, 0, 0, 0]


In [13]:
errors = 0

for a, b in zip(sifted_alice_key, sifted_bob_key):
    if a != b:
        errors += 1

if len(sifted_alice_key) > 0:
    error_rate = errors / len(sifted_alice_key)
else:
    error_rate = 0

print("Number of sifted bits:", len(sifted_alice_key))
print("Number of errors:", errors)
print("Error rate:", error_rate)

if error_rate == 0:
    print("No attack detected. Alice and Bob have the same sifted key.")
else:
    print("Errors detected.")

Number of sifted bits: 4
Number of errors: 0
Error rate: 0.0
No attack detected. Alice and Bob have the same sifted key.


In [14]:
print("BB84 without attacker summary")
print("=" * 50)

print("Alice bits:       ", alice_bits)
print("Alice bases:      ", alice_bases)
print("Bob bases:        ", bob_bases)
print("Bob results:      ", bob_results)
print("Matching indices: ", matching_indices)
print("Alice sifted key: ", sifted_alice_key)
print("Bob sifted key:   ", sifted_bob_key)
print("Error rate:       ", error_rate)

BB84 without attacker summary
Alice bits:        [1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0]
Alice bases:       [1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1]
Bob bases:         [1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0]
Bob results:       [1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0]
Matching indices:  [0, 3, 9, 10]
Alice sifted key:  [1, 0, 0, 0]
Bob sifted key:    [1, 0, 0, 0]
Error rate:        0.0
